In [2]:
# ============================================================
# CHATBOT BANCOESTADO - PRUEBA 1 ING.SOLUCIONES CON IA
# Integracion de los Notebooks de GitHub Models API, LangChain Model API, Streaming y Memory:
# 1) GitHub Models API
# 2) LangChain Model API
# 3) LangChain Streaming
# 4) LangChain Memory
# ============================================================

# ----------------------------
# IMPORTS
# ----------------------------
import os
import time
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# ----------------------------
# 1. GITHUB MODELS API
# ----------------------------
print("=== 1. Configuración GitHub Models API ===")

github_base_url = os.getenv("GITHUB_BASE_URL") or os.getenv("OPENAI_BASE_URL")
github_token = os.getenv("GITHUB_TOKEN")

if not github_base_url:
    raise ValueError("No se encontró GITHUB_BASE_URL ni OPENAI_BASE_URL en las variables de entorno.")

if not github_token:
    raise ValueError("No se encontró GITHUB_TOKEN en las variables de entorno.")

client = OpenAI(
    base_url=github_base_url,
    api_key=github_token
)

print("Base URL:", github_base_url)
print("API Key configurada:", "✓" if github_token else "✗")

# Prueba directa con OpenAI Client
print("\n=== Prueba directa con GitHub Models API ===")
try:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "Eres un asistente virtual de BancoEstado. "
                    "Respondes preguntas generales sobre cuentas, tarjetas, transferencias "
                    "y servicios bancarios. No solicites datos sensibles como RUT, claves, "
                    "coordenadas o números completos de tarjeta."
                )
            },
            {
                "role": "user",
                "content": "¿Qué debo hacer si pierdo mi tarjeta?"
            }
        ],
        temperature=0.2,
        max_tokens=180
    )

    print("Respuesta directa:")
    print(response.choices[0].message.content)
    print(f"Modelo usado: {response.model}")
    print(f"Tokens usados: {response.usage.total_tokens}")

except Exception as e:
    print("Error en la llamada directa:", e)

# ----------------------------
# 2. LANGCHAIN MODEL API
# ----------------------------
print("\n=== 2. Configuración LangChain Model API ===")

llm = ChatOpenAI(
    base_url=github_base_url,
    api_key=github_token,
    model="gpt-4o",
    temperature=0.2,
    max_tokens=250
)

print("Modelo LangChain configurado:", llm.model_name)

# ----------------------------
# 3. LANGCHAIN MEMORY
# ----------------------------
print("\n=== 3. Configuración de memoria conversacional ===")

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente virtual de BancoEstado. "
        "Tu función es responder preguntas frecuentes sobre cuentas, tarjetas, transferencias "
        "y servicios bancarios de forma clara, breve, formal y segura. "
        "Nunca pidas datos sensibles como RUT, contraseñas, claves, coordenadas o números completos de tarjeta. "
        "Si la consulta requiere validación personal o información no disponible, indica que el cliente debe usar canales oficiales."
    ),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

chain = prompt | llm

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

conversation = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history"
)

# ----------------------------
# 4. LANGCHAIN STREAMING
# ----------------------------
print("\n=== 4. Streaming habilitado ===")

def responder_con_streaming_y_memoria(pregunta: str, session_id: str):
    history = get_session_history(session_id)

    mensajes = [
        SystemMessage(content=
            "Eres un asistente virtual de BancoEstado. "
            "Responde preguntas frecuentes sobre cuentas, tarjetas, transferencias "
            "y servicios bancarios de forma clara, breve, formal y segura. "
            "Nunca pidas datos sensibles como RUT, contraseñas, claves, coordenadas o números completos de tarjeta. "
            "Si la consulta requiere validación personal o información no disponible, deriva a canales oficiales."
        )
    ] + history.messages + [HumanMessage(content=pregunta)]

    respuesta_texto = ""
    for chunk in llm.stream(mensajes):
        contenido = chunk.content
        print(contenido, end="", flush=True)
        respuesta_texto += contenido
        time.sleep(0.1)

    history.add_user_message(pregunta)
    history.add_ai_message(respuesta_texto)

    return respuesta_texto

# ----------------------------
# CHATBOT INTERACTIVO
# ----------------------------
print("\n=== CHATBOT INTERACTIVO BANCOESTADO ===")
print("Escribe tu consulta y presiona Enter.")
print("Para terminar, escribe: salir\n")

session_id = "bancoestado_chat"

while True:
    pregunta = input("Tú: ").strip()

    if pregunta.lower() == "salir":
        print("Chatbot: Hasta luego. Gracias por usar el asistente virtual de BancoEstado.")
        break

    if not pregunta:
        print("Chatbot: Por favor, escribe una consulta válida.")
        continue

    print("Chatbot: ", end="", flush=True)

    try:
        responder_con_streaming_y_memoria(pregunta, session_id)
        print("\n")

    except Exception as e:
        print(f"\nError: {e}")
        print("Verifica tu configuración, conexión o credenciales.\n")

# ----------------------------
# MOSTRAR HISTORIAL FINAL
# ----------------------------
print("\n=== HISTORIAL DE LA CONVERSACIÓN ===")
historial = store.get(session_id, InMemoryChatMessageHistory()).messages

if historial:
    for i, msg in enumerate(historial, start=1):
        print(f"{i}. {msg.type}: {msg.content}")
else:
    print("No se registró historial.")

=== 1. Configuración GitHub Models API ===
Base URL: https://models.inference.ai.azure.com
API Key configurada: ✓

=== Prueba directa con GitHub Models API ===
Respuesta directa:
Si pierdes tu tarjeta de BancoEstado, es importante actuar rápidamente para proteger tus fondos. Aquí te indico los pasos que debes seguir:

1. **Bloqueo de la tarjeta**: 
   - Puedes bloquear tu tarjeta llamando al número de atención al cliente de BancoEstado: 600 200 7000. 
   - También puedes hacerlo desde la aplicación móvil de BancoEstado o en el sitio web oficial, ingresando a tu cuenta y seleccionando la opción para bloquear tarjetas.

2. **Solicitar reposición**:
   - Una vez bloqueada, puedes solicitar una nueva tarjeta. Esto lo puedes hacer en cualquier sucursal de BancoEstado o a través de los canales digitales, dependiendo del tipo de tarjeta que tengas.

3. **Revisar movimientos**:
   - Es recomendable revisar los movimientos de tu cuenta para asegurarte de que no se hayan realizado transacciones 

Tú:  ¿Como creo una cuenta rut?


Chatbot: Para crear una CuentaRUT en BancoEstado, puedes hacerlo de manera sencilla siguiendo estos pasos:

1. **Requisitos**: 
   - Ser mayor de 12 años (mujeres) o 14 años (hombres).
   - Tener cédula de identidad chilena vigente.

2. **Canales de apertura**:
   - **En línea**: Ingresa al sitio web oficial de BancoEstado ([www.bancoestado.cl](http://www.bancoestado.cl)) o utiliza la aplicación móvil BancoEstado. Busca la opción "Abrir CuentaRUT" y sigue las instrucciones.
   - **Presencial**: Acude a cualquier sucursal de BancoEstado o BancoEstado Express con tu cédula de identidad vigente.

3. **Activación**: Una vez creada, deberás activarla y retirar tu tarjeta en la sucursal indicada, si corresponde.

Si necesitas más información o asistencia, te recomiendo contactar directamente a BancoEstado a través de su sitio web oficial, aplicación móvil o llamando al número de atención al cliente 600 200 7000.



Tú:  salir


Chatbot: Hasta luego. Gracias por usar el asistente virtual de BancoEstado.

=== HISTORIAL DE LA CONVERSACIÓN ===
1. human: ¿Como creo una cuenta rut?
2. ai: Para crear una CuentaRUT en BancoEstado, puedes hacerlo de manera sencilla siguiendo estos pasos:

1. **Requisitos**: 
   - Ser mayor de 12 años (mujeres) o 14 años (hombres).
   - Tener cédula de identidad chilena vigente.

2. **Canales de apertura**:
   - **En línea**: Ingresa al sitio web oficial de BancoEstado ([www.bancoestado.cl](http://www.bancoestado.cl)) o utiliza la aplicación móvil BancoEstado. Busca la opción "Abrir CuentaRUT" y sigue las instrucciones.
   - **Presencial**: Acude a cualquier sucursal de BancoEstado o BancoEstado Express con tu cédula de identidad vigente.

3. **Activación**: Una vez creada, deberás activarla y retirar tu tarjeta en la sucursal indicada, si corresponde.

Si necesitas más información o asistencia, te recomiendo contactar directamente a BancoEstado a través de su sitio web oficial, aplica